In [1]:
from langchain.schema import Document

def chunk_cvs(texts: list[str], metadatas: list[dict], chunk_size: int = 400, overlap: int = 50) -> list[Document]:
    assert len(texts) == len(metadatas), "texts and metadatas must have the same length"
    assert 0 <= overlap < chunk_size, "overlap must be non-negative and smaller than chunk_size"

    def split_into_chunks(text: str) -> list[str]:
        words = text.split()
        chunks = []
        start = 0
        while start < len(words):
            end = start + chunk_size
            # If it's the final chunk and too small, merge with previous
            if len(words) - start < chunk_size // 2 and chunks:
                chunks[-1].extend(words[start:])
                break
            chunk = words[start:end]
            chunks.append(chunk)
            if end >= len(words): break
            start += chunk_size - overlap
        return [' '.join(chunk) for chunk in chunks]

    all_docs = []
    for text, metadata in zip(texts, metadatas):
        for chunk in split_into_chunks(text):
            all_docs.append(Document(page_content=chunk, metadata=metadata.copy()))
    
    return all_docs


In [4]:
texts = [
    """This is the first CV. It contains multiple
    
    sentences for testing the chunking logic.""",
    "Another CV with a different structure and more content to ensure the logic generalizes well."
]
metadatas = [
    {"filename": "cv_1.txt"},
    {"filename": "cv_2.txt"}
]

chunks = chunk_cvs(texts, metadatas, chunk_size=10, overlap=3)
for doc in chunks:
    print(f"Chunk ({doc.metadata['filename']}): {doc.page_content}\n")


Chunk (cv_1.txt): This is the first CV. It contains multiple sentences for

Chunk (cv_1.txt): multiple sentences for testing the chunking logic.

Chunk (cv_2.txt): Another CV with a different structure and more content to

Chunk (cv_2.txt): more content to ensure the logic generalizes well.

